In [1]:
using Flux
using Enzyme
using Plots
import .EnzymeRules: forward, reverse, augmented_primal
using .EnzymeRules

In [10]:
function dfc(Rij::Float32, Rc::Float32)
    if Rij >= Rc
        return 0.0f0
    end
    ε = eps(Float32)
    denom = 1 - (Rij / Rc)^2
    if denom < ε
        return 0.0f0
    end
    arg = 1 - 1 / denom
    return exp(arg) * (2 * Rij / (Rc^2 * denom^2))
end
function import_all()
    include("../src/Data_prep.jl")
    include("../src/Utils.jl")
    include("../src/Train.jl")
    include("../src/Model.jl")
    include("../src/Loss.jl")
end

import_all()

loss_train (generic function with 1 method)

In [3]:
model = G1Layer(2 , 5.0f0 , 3.0f0)

x = rand(Float32 , 1 , 2 )
model(x)

2×1 Matrix{Float32}:
 0.5965442
 0.22860672

In [9]:
function forward(
    config::FwdConfigWidth{2 , false , true , true , false}, 
    func::Const{G1Layer}, 
    ::Type{<:BatchDuplicatedNoNeed}, 
    x::BatchDuplicated{Matrix{Float32}, 2}
)
    println("custom batch  rule is being used to compute this derivative")
    n_batch = size(x.val , 1)

    out = model(x.val)

    println(typeof(out))

    
    d_out = Tuple(out * 10 for _ in 1:n_batch)

    return BatchDuplicated(out , d_out)
end




Enzyme.gradient(set_runtime_activity(Forward) , xx -> model(xx)[1] , x)


LoadError: MethodError: no method matching forward(::FwdConfigWidth{2, true, true, true, false}, ::Const{G1Layer}, ::Type{BatchDuplicated{Matrix{Float32}, 2}}, ::BatchDuplicated{Matrix{Float32}, 2})

[0mClosest candidates are:
[0m  forward(::FwdConfig, [91m::Const{typeof(sort!)}[39m, ::Type{<:Union{BatchDuplicated, BatchDuplicatedNoNeed, Const}}, ::BatchDuplicated{T, N}; kwargs...) where {T<:(AbstractArray{<:AbstractFloat}), N}
[0m[90m   @[39m [35mEnzyme[39m [90mC:\Users\Gianmarco\.julia\packages\Enzyme\LMVya\src\[39m[90m[4minternal_rules.jl:936[24m[39m
[0m  forward(::FwdConfig, [91m::Const{typeof(deepcopy)}[39m, ::Type{<:BatchDuplicated}, ::BatchDuplicated{T, N}) where {T, N}
[0m[90m   @[39m [35mEnzyme[39m [90mC:\Users\Gianmarco\.julia\packages\Enzyme\LMVya\src\[39m[90m[4minternal_rules.jl:205[24m[39m
[0m  forward(::FwdConfig, ::Const{G1Layer}, [91m::Type{<:BatchDuplicatedNoNeed}[39m, ::BatchDuplicated{Matrix{Float32}, 2})
[0m[90m   @[39m [36mMain[

In [ ]:
Enzyme.jacobian(Forward , xx -> model(xx) , x)

In [12]:
model = G1Layer(2 , 5.0f0 , 2.0f0)

x = rand(Float32 , 1 , 20)

y = sum(rand(Float32 , 20) .* x')

function losss(m , x , y)

    e_L = mean((m(x) .- y) .^2)
    f_L = Enzyme.gradient(Reverse , (mm,xx) -> mm(xx)[1], Const(m) , x)[2]

    return e_L + norm(f_L)
end

o =  OptimiserChain(ClipNorm(1.0), Adam(0.1))

    # Setup dell'ottimizzatore con il modello
opt = Flux.setup(o, model)
grad = Enzyme.gradient(set_runtime_activity(Reverse) , (m , xx , yy)-> losss(m , xx ,yy) , model , Const(x) , Const(y))[1]

Flux.update!(opt, model, grad)

┌ Warning: Using fallback BLAS replacements for (["sasum_64_"]), performance may be degraded
└ @ Enzyme.Compiler C:\Users\Gianmarco\.julia\packages\Enzyme\Qi7lx\src\compiler.jl:3659


Enzyme.Compiler.EnzymeNoTypeError: Enzyme cannot deduce type
Current scope: 
; Function Attrs: mustprogress nofree willreturn
define internal fastcc void @preprocess_diffejulia_G1Layer_6885({ {} addrspace(10)*, {} addrspace(10)*, float, float } addrspace(11)* nocapture nofree noundef nonnull readonly align 8 dereferenceable(24) "enzyme_type"="{[-1]:Pointer, [-1,0]:Pointer, [-1,0,0]:Pointer, [-1,0,0,-1]:Float@float, [-1,0,8]:Integer, [-1,0,9]:Integer, [-1,0,10]:Integer, [-1,0,11]:Integer, [-1,0,12]:Integer, [-1,0,13]:Integer, [-1,0,14]:Integer, [-1,0,15]:Integer, [-1,0,16]:Integer, [-1,0,17]:Integer, [-1,0,18]:Integer, [-1,0,19]:Integer, [-1,0,20]:Integer, [-1,0,21]:Integer, [-1,0,22]:Integer, [-1,0,23]:Integer, [-1,0,24]:Integer, [-1,0,25]:Integer, [-1,0,26]:Integer, [-1,0,27]:Integer, [-1,0,28]:Integer, [-1,0,29]:Integer, [-1,0,30]:Integer, [-1,0,31]:Integer, [-1,0,32]:Integer, [-1,0,33]:Integer, [-1,0,34]:Integer, [-1,0,35]:Integer, [-1,0,36]:Integer, [-1,0,37]:Integer, [-1,0,38]:Int

In [3]:


#file_path = "../test/reduced_train.xyz"
file_path = "dataset.xyz"

# Step 1: Extract information from the input file
species, unique_species, all_cells, dataset, all_energies = extract_data(file_path)

lattice = all_cells[1]

# Create the neural network input dataset and forces 
nn_input_dataset , all_forces , species_idx = prepare_nn_data(dataset, species, unique_species)

# Step 4: Data preprocessing
x_train , y_train , x_val , y_val , x_test , y_test , mean_and_std = data_preprocess(nn_input_dataset, all_energies, all_forces, split=[0.6, 0.2, 0.2])
println("Preprocessed data")

e_mean = mean_and_std[1]
e_std = mean_and_std[2]

# Step 5 : Build the model for each species 

model = build_species_models(unique_species, species_idx, 5 , 6f0, depth = 1 )

println("All model built")



UndefVarError: UndefVarError: `extract_data` not defined

In [4]:
import_all()
### === Single sample ===
x_sample = x_train[1]
y_sample = y_train[1]




# Model prediction
out_sample = dispatch(x_sample, model)
println("Model output with a single configuration as input: $out_sample")



# Total loss
loss_sample = loss( model , x_sample , y_sample  ; λ = 1.0)
println("Model loss on the sample with a single configuration: $loss_sample")


### === Batch of samples ===
x_batch = x_train[1:3, :]
y_batch = y_train[1:3]

# Model prediction
out_batch = dispatch(x_batch, model)
println("Model output with a batch as input: $out_batch")


# Total loss
loss_batch = loss(model , x_batch , y_batch  ; λ = 1.0)
println("Model loss on the batch: $loss_batch")


UndefVarError: UndefVarError: `import_all` not defined

In [5]:


# Step 7: Train the model

 best_model , train_loss , train_val = train_model!(
    model,
    x_train, y_train,
    x_val, y_val;
   epochs = 1000, forces = false, initial_lr = 5e-4  , min_lr = 10e-6  , lattice = lattice , λ = 1f0)



UndefVarError: UndefVarError: `lattice` not defined

In [6]:


# True energies
e_tr = extract_energies(y_train) * e_std .+ e_mean
e_val = extract_energies(y_val)  * e_std .+ e_mean
e_te = extract_energies(y_test)  * e_std .+ e_mean


predict_tr = dispatch( x_train , best_model ; lattice = lattice)  .* e_std .+ e_mean
predict_val = dispatch( x_val , best_model ; lattice = lattice)   .* e_std .+ e_mean
predict_te = dispatch( x_test , best_model  ; lattice = lattice)   .* e_std .+ e_mean

# Number of atoms
n_at = 40

# RMSE per atom
rmse_tr = sqrt(mean((e_tr .- predict_tr).^2)) / n_at
rmse_val = sqrt(mean((e_val .- predict_val).^2)) / n_at
rmse_te = sqrt(mean((e_te .- predict_te).^2)) / n_at

# Common limits for all plots
emin = ([minimum(e_tr), minimum(e_val), minimum(e_te)])
emax = ([maximum(e_tr), maximum(e_val), maximum(e_te)])

# Plot function
function make_plot(true_vals, preds, rmse, setname , emin , emax)
    scatter(
        true_vals, preds,
        xlabel = "True energy [eV]",
        ylabel = "Predicted energy [eV]",
        title = "$setname set: Energy prediction vs True",
        label = "Predictions",
        legend = :topleft,
        alpha = 0.7,
        markersize = 4,

    )
    # Add bisector
    plot!([emin, emax], [emin, emax], linestyle=:dash, color=:black, label="x=y")

    # Add RMSE textbox (upper right corner)
    annotate!(
        (emax - 0.05*(emax-emin), emax - 0.05*(emax-emin),
         text("RMSE = $(round(1e3 * rmse, digits=2)) meV/atom", :right, 10, "black"))
    )
        annotate!(
        (emax - 0.05*(emax-emin), emax - 0.1*(emax-emin),
         text("Correlation = $(round(cor(true_vals , preds), digits=2))", :right, 10, "black"))
    )
end

# Generate the three plots
p1 = make_plot(e_tr, predict_tr, rmse_tr, "Train" , emin[1] , emax[1])
p2 = make_plot(e_val, predict_val, rmse_val, "Validation" , emin[2] , emax[2])
p3 = make_plot(e_te, predict_te, rmse_te, "Test" , emin[3] , emax[3])

# Se vuoi visualizzarli affiancati
display(p1)
display(p2)
display(p3)

savefig(p1 , "Train.png")
savefig(p2 , "Val.png")
savefig(p3 , "Test.png")

UndefVarError: UndefVarError: `extract_energies` not defined

In [7]:
using Statistics
using Plots

# --- Funzione per RMSE ---
rmse(ŷ, y) = sqrt(mean((ŷ .- y).^2))

# --- Estrazione forze ---
f_tr    = extract_forces(y_train )
f_pr_tr = predict_forces(x_train, best_model) 

f_val    = extract_forces(y_val ) 
f_pr_val = predict_forces(x_val, best_model)

f_te    = extract_forces(y_test)
f_pr_te = predict_forces(x_test, best_model) 

# --- Flatten per confronto ---
f_tr_vec    = reshape(f_tr, :)
f_pr_tr_vec = reshape(f_pr_tr, :)

f_val_vec    = reshape(f_val, :)
f_pr_val_vec = reshape(f_pr_val, :)

f_te_vec    = reshape(f_te, :)
f_pr_te_vec = reshape(f_pr_te, :)

# --- RMSE ---
rmse_tr  = rmse(f_pr_tr_vec, f_tr_vec)
rmse_val = rmse(f_pr_val_vec, f_val_vec)
rmse_te  = rmse(f_pr_te_vec, f_te_vec)

println("RMSE per atom Train = $rmse_tr eV / Å")
println("RMSE per atom Val   = $rmse_val eV / Å")
println("RMSE per atom Test  = $rmse_te eV / Å")

# --- Funzione per scatter + salvataggio ---
function plot_forces(y_true, y_pred, filename; title="")
    minval = min(minimum(y_true), minimum(y_pred))
    maxval = max(maximum(y_true), maximum(y_pred))
    p = scatter(y_true, y_pred,
                markersize=2, alpha=0.6,
                xlabel="Forze Reali", ylabel="Forze Predette",
                title=title, legend=false)
    plot!(p, [minval, maxval], [minval, maxval], c=:red, lw=2, label="")
    savefig(p, filename)
    return p
end

# --- Plots separati ---
plot_forces(f_tr_vec, f_pr_tr_vec,
            "forces_train.png",
            title="Train (RMSE=$(round(rmse_tr,digits=4)))")

plot_forces(f_val_vec, f_pr_val_vec,
            "forces_val.png",
            title="Validation (RMSE=$(round(rmse_val,digits=4)))")

plot_forces(f_te_vec, f_pr_te_vec,
            "forces_test.png",
            title="Test (RMSE=$(round(rmse_te,digits=4)))")

println("finito")


UndefVarError: UndefVarError: `extract_forces` not defined